In [ ]:
library(DESeq2)
library(tximport)
library(readr)
library(pheatmap)
library(tidyverse)
library(RColorBrewer)
library(clusterProfiler)
library(org.Mm.eg.db)
library(ggrepel)
library(svglite)
library(gmodels)

## Input

In [ ]:
# tximport import
path_root <- "/path_to/RNAseq_data"

sampleF <- read.table(file.path(path_root, "02_counts", "sample.tsv")) 

files <- file.path(path_root, "02_counts", sampleF$V1)
names(files) <- substr(sampleF$V1, 1, 10)

txi.rsem <- tximport(files, type = "rsem", txIn = FALSE, txOut = FALSE)
txi.rsem$length[txi.rsem$length == 0] <- 1

# design
condition <- factor(c("control","control","treat","treat","treat","treat"))
sampleTable <- data.frame(row.names=colnames(txi.rsem$counts), condition)

# DESeq2
dds <- DESeqDataSetFromTximport(txi.rsem, sampleTable, ~ condition)
dds@design

dds <- DESeq(dds)

res <- results(dds, contrast = c("condition", "treat", "control"))
res <- res[order(abs((res$padj))),]

##  Corr

In [ ]:
FPCA_data <- txi.rsem$abundance
pca.info <- fast.prcomp(FPCA_data)
pca.data <- data.frame(sample = rownames(pca.info$rotation),Type = c(rep("Control",2),rep("Treatment",4)),
                        pca.info$rotation)

ggplot() +
  theme_bw(base_size = 20) +
  geom_point(pca.data, mapping = aes(x = PC1, y = PC2, color = Type, size = 15)) +
  geom_text_repel(pca.data, mapping = aes(x = PC1, y = PC2, label = sample)) +
  theme(axis.line = element_line(color = "black")) +
  theme(axis.text = element_text(color = "black")) +
  theme(axis.ticks = element_line(color = "black")) +
  scale_color_manual(values = c("#00dae0", "#ff9289"))

plot_out <- file.path(path_root, "01_plot", "sample_pca.svg")
ggsave(plot_out, width = 6.5, height = 5)

## HeatMap

In [ ]:
library(ComplexHeatmap)
library(circlize)

In [ ]:
path_root <- "/path_to/RNAseq_data"

Gene_List <- c("Vegfa", "Gdf15", "Il34", "Areg", "Plaur", "Ptgs1", "Gsn", "Grn", "Lgals3", "Angptl6", 
                "Mmp11", "Nt5e", "Sox4", "Ndrg1", "Epha2", "Vash2", "Pfkfb4", 
                
                "Thbs1", "Dlc1", "Wif1", "Sema3f", "Lect2", "Lifr", "Cx3cl1", "Irf1", "Tnfsf10", "Rassf1", 
                "Gadd45b", "Tgfbr3", "Abca1", "Sh2d4a", "Mat1a"
                )


## All Genes
DEG_df <- res %>% as.data.frame() %>% arrange(-log2FoldChange) %>% filter(padj <= 0.05) %>% 
                  filter(abs(log2FoldChange) >= 1)

Up_Genes <- DEG_df %>% filter(log2FoldChange >= 1) %>% rownames()
Down_Genes <- DEG_df %>% filter(log2FoldChange <= -1) %>% rownames()

choose_gene <- c(Down_Genes, Up_Genes)
choose_matrix <- as.data.frame(txi.rsem$counts)[choose_gene, ]
choose_matrix <- t(scale(t(choose_matrix)))

## Get index of genes of interest
index <- which(rownames(choose_matrix) %in% Gene_List)
labs <- rownames(choose_matrix)[index]

lab2 <- rowAnnotation(foo = anno_mark(at = index,
                     labels = labs, labels_gp = gpar(fontsize = 8), 
                     lines_gp = gpar()))
col_fun = colorRamp2(c(-2, 0, 2), c("#317cb6","white", "#b52330"))

## plot and save
plot_out <- file.path(path_root, "01_plot", "HeatMap_DEG.All.svg")
svglite(plot_out, width = 6, height = 7)

Heatmap(choose_matrix, row_names_gp = gpar(fontsize = 6), cluster_rows = FALSE, show_row_names = FALSE, col = col_fun,
        cluster_columns = FALSE, show_column_names = FALSE,
        column_names_gp = gpar(fontsize = 8), name = "Exp", right_annotation = lab2)

dev.off()

## DEG volcanol

In [ ]:
path_root <- "/path_to/RNAseq_data"

rs <- res %>% as.data.frame()
rs <- rs[rowSums(is.na(rs))==0,]
rs <- rs %>% filter(!is.na(log2FoldChange))
rs$gene <- row.names(rs)

rs$type <- "No change"
rs[rs$log2FoldChange > 1 & rs$padj <= 0.05,]$type <- "Up"
rs[rs$log2FoldChange < -1 & rs$padj <=0.05,]$type <- "Down"
rs$type <- as.factor(rs$type)

ggplot(rs, aes(log2FoldChange, -log10(padj), color = type)) +
    theme_bw(base_size = 20) +
    geom_point(size = 0.5) +
    scale_color_manual(values = c("#546de5", "#d2dae2","#ff4757")) + 
    theme(panel.grid = element_blank(), panel.border = element_blank()) +
    theme(axis.line = element_line(size=1, colour = "black")) + 
    theme(axis.text = element_text(colour = "black")) +
    theme(legend.title = element_blank()) +
    geom_vline(xintercept=c(-1,1), lty=4, col="black", lwd=0.5)


plot_out <- file.path(path_root, "01_plot", "sample_volcanol.svg")
ggsave(plot_out, width = 7, height = 5)

## GSEA

### GseaVis Plottie

In [ ]:
library(msigdbr)
library(enrichplot)
library(GSEABase)
library(dplyr)
library(GseaVis)

In [ ]:
## Gene rank
diff_Gene <- rs
diff_Gene$Gene <- rownames(diff_Gene)

gene <- diff_Gene$Gene
entrezid <- bitr(gene, fromType = "SYMBOL",toType = "ENTREZID",OrgDb = "org.Mm.eg.db")

Fina_data <- merge(entrezid, diff_Gene, by.x = "SYMBOL", by.y = "Gene")

ranks <- Fina_data$log2FoldChange
names(ranks) <- Fina_data$ENTREZID
ranks = sort(ranks, decreasing = T)


path_root <- "/path_to/02_RNAseq"
GMT_FilePath <- "/path_to/Data_ref/msigdb_Data/Mouse_data/msigdb_v2025.1.Mm_GMTs"
GMT_fileList <- c(
  "MH" = "mh.all.v2025.1.Mm.entrez.gmt"
)

for(GMTNM in names(GMT_fileList)){

  GMT_FileNM <- GMT_fileList[[GMTNM]]

  m_df1 <- msigdbr(species = "Mus musculus", category = "H")
  mlist_HALLmark <- as.data.frame(m_df1 %>% dplyr::select(gs_name, entrez_gene))

  HALL <- GSEA(ranks, TERM2GENE = mlist_HALLmark, minGSSize = 10, maxGSSize = 1000, pvalueCutoff=1)

  file_out <- file.path(path_root, "03_result", "04_GSEA", paste0("DEG_GSEA.", GMTNM, ".tsv"))
  HALL_out <- HALL@result

  write.table(HALL_out, file_out, sep = "\t", row.names = FALSE)
}

In [ ]:
HALL2 <- setReadable(HALL, OrgDb="org.Mm.eg.db", keyType="ENTREZID")

GMT_FilePath <- "/path_to/Data_ref/msigdb_Data/Mouse_data/msigdb_v2025.1.Mm_GMTs"
GMT_FileNM <- "mh.all.v2025.1.Mm.symbols.gmt"
gmt_list <- getGmt(file.path(GMT_FilePath, GMT_FileNM))
gmt_df <- do.call(rbind, lapply(names(gmt_list), function(set_name) {
  data.frame(
    gene_set = set_name,
    gene = geneIds(gmt_list[[set_name]]),
    stringsAsFactors = FALSE
  )
}))
names(gmt_df) <- c("gs_name", "Symbol")
gmt_list <- split(gmt_df$Symbol, gmt_df$gs_name)

###########################
## Gene rank
diff_Gene <- rs
diff_Gene$Gene <- rownames(diff_Gene)

gene <- diff_Gene$Gene
entrezid <- bitr(gene, fromType = "SYMBOL",toType = "ENTREZID",OrgDb = "org.Mm.eg.db")

Fina_data <- merge(entrezid, diff_Gene, by.x = "SYMBOL", by.y = "Gene")

ranks <- Fina_data$log2FoldChange
names(ranks) <- Fina_data$SYMBOL
ranks = sort(ranks, decreasing = T)

HALL2@geneSets <- gmt_list
HALL2@geneList <- ranks

In [ ]:
GSEA_Terms <- c("HALLMARK_INTERFERON_GAMMA_RESPONSE", "HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_ANGIOGENESIS")

for (GseaNM in GSEA_Terms){

    P1 <- gseaNb(object = HALL2, geneSetID = GseaNM, addPval = T)
    plot_out <- file.path(path_root, "01_plot", paste0("GSEA_", gsub("HALLMARK_", "", GseaNM), ".svg"))
    ggsave(plot = P1, plot_out, width = 6, height = 6)

}

### Add Genes

In [ ]:
mygene <- c("Egfr", "Ets1", "Siah2", "Pim1", "Ddit4", "Kdm3a", 
              "Atp7a", "Maff", "Jun", "Fos", "Wsb1", "S100a4", "Ppfia4", 
              "Foxo3", "Plaur", "Jun", "Vegfa", "Irs2", "Rora", "Klf6", "Atf3", "Ndrg1") 


# plot
gseaNb(object = HALL2,
       geneSetID = 'HALLMARK_HYPOXIA',
       addGene = mygene, 
       arrowType = 'open', addPval = T
       )

plot_out <- file.path(path_root, "01_plot", "GSEA_HYPOXIA_MarkGenes.svg")
ggsave(plot_out, width = 6, height = 6)

## GO Enrichment net

In [ ]:
library(aPEAR)
library(cols4all)

CountsFi <- txi.rsem$counts[rowSums(txi.rsem$counts) > 6, ]
CountsFiGenes <- row.names(CountsFi)

Data_in <- rs %>% filter(baseMean > 100)
Markers_ID <- bitr(Data_in$gene, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = "org.Mm.eg.db")
names(Markers_ID) <- c("gene", "ENTRZID")

Data_AddEntrzid <- merge(Data_in, Markers_ID, by = "gene", all.y = TRUE)


Data_AddEntrzid <- na.omit(Data_AddEntrzid)
Data_AddEntrzid <- Data_AddEntrzid %>% arrange(desc(log2FoldChange)) %>% dplyr::select(ENTRZID, log2FoldChange)

FCgenelist <- Data_AddEntrzid$log2FoldChange
names(FCgenelist) <- Data_AddEntrzid$ENTRZID

In [ ]:
enrich <- gseGO(FCgenelist, OrgDb = org.Mm.eg.db, ont = 'BP')

enrich <- setReadable(enrich, "org.Mm.eg.db")

file_out <- file.path(path_in, "04_GSEAcell", paste0(CellType, "_GO_GSEA.tsv"))
write.table(enrich@result, file_out, sep= "\t", quote = FALSE, row.names = FALSE)

enrichmentNetwork(enrich@result,
                       colorBy = 'NES',
                       colorType = c("nes"),
                       nodeSize = 'setSize',
                       fontSize = 2,
                       verbose = TRUE) +
    scale_color_continuous_c4a_div('benedictus', reverse = T)

plot_out <- file.path(path_root, "01_plot", "GO_GSEA.svg")
ggsave(plot_out, width = 10, height = 10)

## GSEA bulk/sc RNA-seq HeatMap

In [ ]:
library(ComplexHeatmap)
library(circlize)
library(tidyverse)
library(ggplot2)
library(tidyr)

path_root <- "/path_to/RNA_GSEAHeatMap"

file_bulk <- file.path(path_root, "01_preData", "RNAseq_GSEA_HALLMARK.MH.tsv")
data_bulk <- read.table(file_bulk, sep = "\t", header = TRUE)
data_bulk <- data_bulk[, c("ID", "NES")]
data_bulk$ID <- tolower(data_bulk$ID)
data_bulk$ID <- gsub("_", " ", gsub("hallmark_", "", data_bulk$ID))
names(data_bulk) <- c("Description", "NES")
data_bulk$Source <- "Bulk RNA-seq"

file_sc <- file.path(path_root, "01_preData", "Malignant_H_GSEA.tsv")
data_sc <- read.table(file_sc, sep = "\t", header = TRUE)
data_sc <- data_sc[, c("Description", "NES")]
data_sc$Description <- gsub("_", " ", gsub("hallmark_", "", data_sc$Description))
data_sc$Source <- "scRNA-seq"

Data_merge <- rbind(data_bulk, data_sc)
Data_merge <- pivot_wider(data = Data_merge, names_from = Description, values_from = NES) %>% 
              as.data.frame() %>% column_to_rownames("Source")

In [ ]:
plot_out <- file.path(path_root,  "GSEA_HeatMap_vertical.svg")
svglite(plot_out, width = 4, height = 8)

col_fun = colorRamp2(c(-2, 0, 2), c("#317cb6","white", "#b52330"))

Heatmap(t(Data_merge), row_names_gp = gpar(fontsize = 9), cluster_rows = TRUE, show_row_names = TRUE, col = col_fun,
        cluster_columns = FALSE, show_column_names = TRUE,
        column_names_gp = gpar(fontsize = 9), name = "NES")

dev.off()